# Summarizing pupillometric data

In [1]:
import pypillometry as pp
import pandas as pd

The `pypillometry` package provides several functions for summarizing datasets. Simply printing a `PupilData`, `GazeData` or `EyeData` object gives a readable summary of the main properties of the dataset and also prints the complete history of operations applied to it.

In [2]:
d = pp.get_example_data("rlmw_002_short")
d.pupil_blinks_detect(apply_mask=True)
print(d)

EyeData(test short, 2.0MiB):
 n                 : 20465
 sampling_rate     : 500.0
 eyes              : ['left', 'right']
 data              : ['left_x', 'left_y', 'left_pupil', 'right_x', 'right_y', 'right_pupil']
 nevents           : 40
 experimental_setup: {'screen_resolution': (1280, 1024), 'physical_screen_size_mm': (300.0, 200.0), 'eye_to_screen_distance_mm': np.float64(600.0), 'camera_position': 'not set'}
 calibration       : not available
 duration_minutes  : 0.6821666666666667
 start_min         : 0.0
 end_min           : 0.6821333333333333
 parameters        : {}
 glimpse           : EyeDataDict(vars=6,n=20465,shape=(20465,)): 
  left_x (float64): 655.6, 655.7, 655.0, 654.5, 655.0...
  left_y (float64): 599.9, 598.9, 597.6, 597.8, 597.8...
  left_pupil (float64): 1121.0, 1122.0, 1124.0, 1126.0, 1126.0...
  right_x (float64): 773.1, 773.8, 774.8, 776.3, 776.7...
  right_y (float64): 594.2, 593.9, 596.5, 597.4, 597.3...
  right_pupil (float64): 949.0, 951.0, 956.0, 959.0, 960.

By calling the function `summary()`, the same information is returned as a `dict`:

In [3]:
d.summary()

{'name': 'test short',
 'n': 20465,
 'sampling_rate': np.float64(500.0),
 'eyes': ['left', 'right'],
 'data': ['left_x',
  'left_y',
  'left_pupil',
  'right_x',
  'right_y',
  'right_pupil'],
 'nevents': 40,
 'experimental_setup': {'screen_resolution': (1280, 1024),
  'physical_screen_size_mm': (300.0, 200.0),
  'eye_to_screen_distance_mm': np.float64(600.0),
  'camera_position': 'not set'},
 'calibration': 'not available',
 'duration_minutes': np.float64(0.6821666666666667),
 'start_min': np.float64(0.0),
 'end_min': np.float64(0.6821333333333333),
 'parameters': '{}',
 'glimpse': 'EyeDataDict(vars=6,n=20465,shape=(20465,)): \n  left_x (float64): 655.6, 655.7, 655.0, 654.5, 655.0...\n  left_y (float64): 599.9, 598.9, 597.6, 597.8, 597.8...\n  left_pupil (float64): 1121.0, 1122.0, 1124.0, 1126.0, 1126.0...\n  right_x (float64): 773.1, 773.8, 774.8, 776.3, 776.7...\n  right_y (float64): 594.2, 593.9, 596.5, 597.4, 597.3...\n  right_pupil (float64): 949.0, 951.0, 956.0, 959.0, 960.0...\

Using `pandas`, this allows you to quickly summarize properties of many datasets in a table.

To illustrate, we load a few datasets (in a real study, there would be one per subject) and detect blinks in each of them:

In [4]:
datasets = [pp.get_example_data("rlmw_002_short"),
            pp.PupilData.from_file("../../data/test.pd"),
            pp.PupilData.from_file("../../data/example_shortblinks.pd")]
for d in datasets:
    d.pupil_blinks_detect(apply_mask=True)

We can now call `summary()` for each of the objects and combine the resulting `dict`s into a table. Here we keep only some of the columns:

In [5]:
tab = pd.DataFrame([d.summary() for d in datasets])
tab[["name", "n", "sampling_rate", "duration_minutes", "nevents", "nblinks"]]

,name,n,sampling_rate,duration_minutes,nevents,nblinks
0,test short,20465,500.0,0.682167,40,"{'left_pupil': 15, 'right_pupil': 13}"
1,test_ro_ka_si_hu,60001,500.0,2.000033,56,{'left_pupil': 71}
2,55910_1_wu,12501,500.0,0.416700,10,{'left_pupil': 22}


Some entries are dictionaries with one entry per eye. For blinks, `blink_stats()` gives more details, e.g., the number and mean duration of blinks per eye:

In [6]:
pd.DataFrame([{"name": d.name, "eye": eye, **stats}
              for d in datasets
              for eye, stats in d.blink_stats().items()])

,name,eye,n,mean,sd,min,max,total_duration,units
0,test short,left,15,354.533333,562.544442,48.0,2278.0,5318.0,ms
1,test short,right,13,138.000000,84.856440,44.0,404.0,1794.0,ms
2,test_ro_ka_si_hu,left,71,556.507042,1016.559662,24.0,5394.0,39512.0,ms
3,55910_1_wu,left,22,249.181818,68.717231,184.0,538.0,5482.0,ms
